In [1]:
import json
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Efecto del hidrógeno — caudal +25%

Comparación controlada de las simulaciones guardadas: propiedades constantes actuales
frente a propiedades de referencia dependientes de temperatura, con los demás parámetros
idénticos. Se muestran Termopar 9 (rollo 2, r=0.309 m, z=0.636 m) y Termopar 11
(rollo 1, r=0.312 m, z=0.636 m). r es distancia desde la superficie interior.
No se ejecuta el simulador; se leen los archivos de resultados.

El modelo de referencia usa H₂ normal, p absoluta=101325 Pa y caudal volumétrico **real**
constante. Conductividad: Assael; viscosidad: Muzny; densidad: Leachman, vía CoolProp.
Se interpolan tablas cada 0.25 K. El perfil supera 1000 K: la extensión hasta 1100 K
se identifica como extrapolación. El modelo también cambia la conducción radial por
la conductividad del gas; no se trata de un cambio exclusivamente convectivo.

**Caudal real: 24.25 m³/s = 19.4 × 1.25.** Ambos modelos usan este caudal.


## Settings

In [2]:

# Works with a kernel started at the repository root or inside experiments/.
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "heattransfer/models").is_dir())
study_directory = ROOT / "experiments/results/hydrogen_flow_plus25_2026-11-06"
thermocouples = [9, 11]
colors = {"baseline": "#1f77b4", "scientific": "#d95f02",
          "experimental": "#2ca02c", "gas": "#666666"}
labels = {"baseline": "H₂ constante", "scientific": "H₂ de referencia"}
plot_height = 850
runs, metadata = {}, {}
for case in ["baseline", "scientific"]:
    metadata[case] = json.loads((study_directory / case / "parameters.json").read_text())
    with np.load(study_directory / case / "results.npz", allow_pickle=False) as archive:
        # Load only point histories: full coil fields remain on disk.
        keys = ["times_s", "gas_K"] + [key for n in thermocouples
                for key in [f"termopar_{n}_K", f"experiment_{n}_hours_C"]]
        runs[case] = {key: archive[key] for key in keys}

for key in ["configuration_sha256", "material", "coils", "points", "timestep_seconds", "duration_seconds", "furnace"]:
    assert metadata["baseline"][key] == metadata["scientific"][key], f"Uncontrolled difference: {key}"
np.testing.assert_array_equal(runs["baseline"]["times_s"], runs["scientific"]["times_s"])
np.testing.assert_array_equal(runs["baseline"]["gas_K"], runs["scientific"]["gas_K"])
hours = runs["baseline"]["times_s"] / 3600

layout = {
    "height": plot_height, "template": "plotly_white",
    "font": {"family": "Arial", "size": 14},
    "margin": {"t": 150, "b": 70, "l": 90, "r": 50},
    "legend": {"orientation": "h", "x": 0, "y": 1.13},
    "hovermode": "x unified",
}

temperature_fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                                subplot_titles=[f"Termopar {n}" for n in thermocouples],
                                vertical_spacing=0.15)
for row, number in enumerate(thermocouples, 1):
    for case in runs:
        temperature_fig.add_trace(go.Scatter(
            x=hours, y=runs[case][f"termopar_{number}_K"], mode="lines",
            name=labels[case], legendgroup=case, showlegend=row==1,
            line={"color": colors[case], "width": 2}), row=row, col=1)
    experiment = runs["baseline"][f"experiment_{number}_hours_C"]
    temperature_fig.add_trace(go.Scatter(
        x=experiment[:,0], y=experiment[:,1]+273.15, mode="lines",
        name="Experimental", legendgroup="experimental", showlegend=row==1,
        line={"color": colors["experimental"], "width": 2, "dash": "dot"}), row=row, col=1)
    temperature_fig.add_trace(go.Scatter(
        x=hours, y=runs["baseline"]["gas_K"], mode="lines",
        name="Gas", legendgroup="gas", showlegend=row==1,
        line={"color": colors["gas"], "dash": "dash"}), row=row, col=1)
    temperature_fig.update_yaxes(title_text="Temperatura [K]", row=row, col=1)
temperature_fig.update_layout(**layout, title={"text": "Efecto de las propiedades de H₂", "x": 0.5})
temperature_fig.update_xaxes(title_text="Tiempo [h]", row=2, col=1);


## Plot — Evolución de temperatura
Simulaciones continuas, mediciones punteadas y temperatura del gas discontinua.

In [3]:
temperature_fig.show()


## Settings — Diferencias
Se interpola cada simulación a los tiempos medidos dentro del intervalo común. No se extrapolan datos. RMSE, MAE y sesgo ponderan igualmente cada muestra experimental.

In [4]:
residuals, metrics = {}, []
for number in thermocouples:
    experiment = runs["baseline"][f"experiment_{number}_hours_C"]
    np.testing.assert_array_equal(experiment, runs["scientific"][f"experiment_{number}_hours_C"])
    overlap = (experiment[:,0] >= hours[0]) & (experiment[:,0] <= hours[-1])
    times = experiment[overlap,0]
    measured = experiment[overlap,1] + 273.15
    if not len(times): raise ValueError(f"No overlapping measurements for Termopar {number}")
    for case in runs:
        difference = np.interp(times, hours, runs[case][f"termopar_{number}_K"]) - measured
        residuals[number,case] = (times,difference)
        metrics.append({"Termopar": number, "Modelo": labels[case], "N": len(times),
                        "RMSE [K]": float(np.sqrt(np.mean(difference**2))),
                        "MAE [K]": float(np.mean(np.abs(difference))),
                        "Sesgo [K]": float(np.mean(difference)),
                        "Máx. |error| [K]": float(np.max(np.abs(difference)))})


## Plot — Simulación menos experimento
Valores positivos significan que la simulación predice mayor temperatura.

In [5]:
difference_fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                                subplot_titles=[f"Termopar {n}" for n in thermocouples],
                                vertical_spacing=0.15)
for row, number in enumerate(thermocouples,1):
    for case in runs:
        times,difference = residuals[number,case]
        difference_fig.add_trace(go.Scatter(
            x=times,y=difference,mode="lines",name=labels[case],legendgroup=case,
            showlegend=row==1,line={"color":colors[case],"width":2}),row=row,col=1)
    difference_fig.add_hline(y=0,line_color="#888888",line_dash="dot",row=row,col=1)
    difference_fig.update_yaxes(title_text="Tsim − Texp [K]",row=row,col=1)
difference_fig.update_layout(**layout,title={"text":"Diferencias frente al experimento","x":0.5})
difference_fig.update_xaxes(title_text="Tiempo [h]",row=2,col=1)
difference_fig.show()


## Plot — Cambio causado por las propiedades
Referencia menos baseline en el mismo punto y tiempo; no es un error frente al experimento.

In [6]:
effect_fig = go.Figure()
for number,color in zip(thermocouples,["#9467bd","#17becf"]):
    effect_fig.add_trace(go.Scatter(x=hours,
        y=runs["scientific"][f"termopar_{number}_K"]-runs["baseline"][f"termopar_{number}_K"],
        mode="lines",name=f"Termopar {number}",line={"color":color}))
effect_fig.update_layout(**{**layout,"height":500},
                         title={"text":"Cambio: H₂ de referencia − H₂ constante","x":0.5},
                         xaxis_title="Tiempo [h]",yaxis_title="ΔT [K]")
effect_fig.add_hline(y=0,line_color="#888888",line_dash="dot")
effect_fig.show()


## Métricas

In [7]:
columns = list(metrics[0])
metrics_fig = go.Figure(go.Table(
    header={"values":columns,"fill_color":"#e8edf4","align":"left"},
    cells={"values":[[round(m[col],3) if isinstance(m[col],float) else m[col] for m in metrics]
                     for col in columns],"align":"left"}))
metrics_fig.update_layout(height=300,margin={"t":20,"b":20,"l":20,"r":20})
metrics_fig.show()
for metric in metrics: print(metric)


{'Termopar': 9, 'Modelo': 'H₂ constante', 'N': 75, 'RMSE [K]': 11.799727911427919, 'MAE [K]': 8.126106785260552, 'Sesgo [K]': 1.9288023820717581, 'Máx. |error| [K]': 34.870399431767964}
{'Termopar': 9, 'Modelo': 'H₂ de referencia', 'N': 75, 'RMSE [K]': 14.213011517044757, 'MAE [K]': 10.916927232110462, 'Sesgo [K]': -7.278995961112265, 'Máx. |error| [K]': 27.828252445566477}
{'Termopar': 11, 'Modelo': 'H₂ constante', 'N': 75, 'RMSE [K]': 15.885390038139796, 'MAE [K]': 11.514808669842518, 'Sesgo [K]': 10.220729884138825, 'Máx. |error| [K]': 42.35256547060487}
{'Termopar': 11, 'Modelo': 'H₂ de referencia', 'N': 75, 'RMSE [K]': 17.335407548445744, 'MAE [K]': 15.457145828035063, 'Sesgo [K]': 2.7259476656454082, 'Máx. |error| [K]': 25.986717149902802}


## Comparación con el caudal original
RMSE frente a las mismas mediciones: 19.4 frente a 24.25 m³/s. Una diferencia negativa indica mejora.

In [8]:
original_metrics = json.loads((ROOT / "experiments/results/hydrogen_2026-11-06/metrics.json").read_text())
print(f"{'Termopar':>9} {'Modelo':>18} {'RMSE 19.4 [K]':>16} {'RMSE 24.25 [K]':>17} {'Cambio [K]':>12}")
flow_comparison = []
for current in metrics:
    previous = next(m for m in original_metrics
                    if (m["Termopar"], m["Modelo"]) == (current["Termopar"], current["Modelo"]))
    change = current["RMSE [K]"] - previous["RMSE [K]"]
    flow_comparison.append({"Termopar":current["Termopar"], "Modelo":current["Modelo"],
                            "RMSE_original_K":previous["RMSE [K]"],
                            "RMSE_plus25_K":current["RMSE [K]"], "change_K":change})
    print(f"{current['Termopar']:9} {current['Modelo']:>18} {previous['RMSE [K]']:16.3f} {current['RMSE [K]']:17.3f} {change:12.3f}")


 Termopar             Modelo    RMSE 19.4 [K]    RMSE 24.25 [K]   Cambio [K]
        9       H₂ constante           15.352            11.800       -3.552
        9   H₂ de referencia           13.799            14.213        0.414
       11       H₂ constante           16.208            15.885       -0.322
       11   H₂ de referencia           10.170            17.335        7.165


## Referencias (APA)

Assael, M. J., Assael, J.-A. M., Huber, M. L., Perkins, R. A., & Takata, Y. (2011). Correlation of the thermal conductivity of normal and parahydrogen from the triple point to 1000 K and up to 100 MPa. *Journal of Physical and Chemical Reference Data, 40*(3), 033101. https://doi.org/10.1063/1.3606499

Leachman, J. W., Jacobsen, R. T., Penoncello, S. G., & Lemmon, E. W. (2009). Fundamental equations of state for parahydrogen, normal hydrogen, and orthohydrogen. *Journal of Physical and Chemical Reference Data, 38*(3), 721–748. https://doi.org/10.1063/1.3160306

Muzny, C. D., Huber, M. L., & Kazakov, A. F. (2013). Correlation for the viscosity of normal hydrogen obtained from symbolic regression. *Journal of Chemical & Engineering Data, 58*(4), 969–979. https://doi.org/10.1021/je301273j

Muzny, C. D., Huber, M. L., & Kazakov, A. F. (2022). Erratum: Correlation for the viscosity of normal hydrogen obtained from symbolic regression. *Journal of Chemical & Engineering Data, 67*, 2855. https://doi.org/10.1021/acs.jced.2c00523
